# Tarea 1 – Parte 1: Recolección y limpieza de noticias peruanas

- **Estudiante:** *(tu nombre y @usuario_github)*
- **Medio elegido:** Opción B – **El Comercio**
- **Fecha de descarga:** 17 y 18 de octubre de 2026
- **URLs base (sitemaps de noticias listados en su `robots.txt`):**
  - `https://elcomercio.pe/sitemap/news/politica/?outputType=xml`
  - `https://elcomercio.pe/sitemap/news/economia/?outputType=xml`
  - `https://elcomercio.pe/sitemap/news/deporte-total/?outputType=xml`
  - `https://elcomercio.pe/sitemap/news/tecnologia/?outputType=xml`

**Método y buenas prácticas.** Se revisó el `robots.txt` (solo bloquea `/catalogo/`, `/buscar/` y el archivo antiguo) y se usó un `User-Agent` descriptivo, pausas de 1.5 s entre peticiones y un único reintento. Los feeds RSS oficiales solo exponían ~15 notas por sección, insuficientes para llegar a 350; por eso se usaron los sitemaps de noticias (60–100 notas por sección, con titular y fecha). La `bajada` se obtuvo de la meta-descripción de cada artículo. El respaldo crudo está en `datos/raw/acumulado.csv`.

In [1]:
import time
import pandas as pd
import plotly.express as px
import plotly.io as pio
import requests

# Si instalaste kaleido, cada gráfico también se guarda como imagen y se ve en GitHub.
try:
    import kaleido  # noqa: F401
    pio.renderers.default = "plotly_mimetype+png"
except ImportError:
    pio.renderers.default = "plotly_mimetype"

UA = "CursoDS-Tarea1/1.0 (proyecto academico; github: TU_USUARIO)"  # el mismo del script
CATEGORIAS = ["politica", "economia", "deportes", "ciencia_tecnologia"]
pd.set_option("display.max_colwidth", 90)

raw = pd.read_csv("datos/raw/acumulado.csv").fillna("")
print("Noticias crudas (filas, columnas):", raw.shape)
raw.head()

Noticias crudas (filas, columnas): (404, 7)


,titular,bajada,medio,seccion_original,url,fecha,feed
0,Facultades legislativas: Senado sigue sin alcanzar consenso y debate continúa entrampado,El pedido de facultades legislativas del Ejecutivo continúa entrampado en el Senado po...,El Comercio,politica,https://elcomercio.pe/politica/facultades-legislativas-senado-sigue-sin-alcanzar-conse...,2026-10-17,politica
1,Renovación Popular se declara en reorganización partidaria: ¿A qué se debe y cuáles so...,"La medida deja sin efecto las credenciales de los coordinadores regionales, provincial...",El Comercio,politica,https://elcomercio.pe/politica/renovacion-popular-se-declara-en-reorganizacion-partida...,2026-10-17,politica
2,"Alfonso, ¿ahora qué?, una crónica de Fernando Vivas sobre López-Chau, líder de Ahora N...","El líder de Ahora Nación y portavoz de su bancada de senadores, Alfonso López-Chau, qu...",El Comercio,politica,https://elcomercio.pe/politica/alfonso-ahora-que-una-cronica-de-fernando-vivas-sobre-l...,2026-10-17,politica
3,Tomás Gálvez dice que Ministerio Público requiere un crédito suplementario: “No tendrí...,"El fiscal de la Nación, Tomás Gálvez, pidió al Gobierno un crédito suplementario de ur...",El Comercio,politica,https://elcomercio.pe/politica/actualidad/tomas-galvez-dice-que-ministerio-publico-req...,2026-10-17,politica
4,De Gregorio Santos a “El lunarejo”: más de 8 mil condenados solicitaron el indulto pre...,"Entre los solicitantes se encuentra Gregorio Santos, exgobernador de Cajamarca, conden...",El Comercio,politica,https://elcomercio.pe/politica/indultos-presidenciales-lista-gracias-presidenciales-ke...,2026-10-17,politica


## 1. Limpieza

Reglas aplicadas, en orden, registrando cuántas noticias elimina cada una:
1. Titulares o URLs vacíos.
2. URLs duplicadas.
3. La nota diaria **"Precio del dólar hoy…"**: es una plantilla que se repite cada día con casi el mismo titular. Si quedaran varias, versiones casi idénticas caerían a la vez en *train* y *test* y inflarían las métricas.
4. Titulares repetidos (ignorando mayúsculas y espacios).

In [2]:
descartes = {}  # motivo -> cantidad
df = raw.copy()

n = len(df)
df = df[(df["titular"].str.strip() != "") & (df["url"].str.strip() != "")]
descartes["titular o URL vacíos"] = n - len(df)

n = len(df)
df = df.drop_duplicates(subset="url")
descartes["URL duplicada"] = n - len(df)

n = len(df)
es_dolar = df["titular"].str.contains(r"precio del d[óo]lar|tipo de cambio del d[óo]lar", case=False, regex=True)
df = df[~es_dolar]
descartes["nota diaria 'Precio del dólar' (casi duplicada)"] = n - len(df)

n = len(df)
titular_norm = df["titular"].str.lower().str.replace(r"\s+", " ", regex=True)
df = df[~titular_norm.duplicated()]
descartes["titular repetido"] = n - len(df)

pd.Series(descartes, name="descartadas").to_frame()

,descartadas
titular o URL vacíos,0
URL duplicada,0
nota diaria 'Precio del dólar' (casi duplicada),6
titular repetido,0


## 2. Mapeo `seccion_original` → categoría final

La categoría se asigna por la **sección que aparece en la URL** de la nota (`seccion_original`). Las secciones sin equivalencia clara en las 4 categorías oficiales se **descartan**; no se crea una quinta categoría.

In [3]:
MAPEO = {
    "politica": "politica",
    "elecciones": "politica",
    "economia": "economia",
    "deporte-total": "deportes",
    "tecnologia": "ciencia_tecnologia",
}
MOTIVO_DESCARTE = {
    "opinion": "columnas de opinión: tema mixto, sin categoría clara",
    "ecdata": "periodismo de datos: tema mixto",
    "eldominical": "revista dominical: tema mixto",
    "ruedas-tuercas": "motores: no encaja en las 4 categorías",
    "bbc-news-mundo": "noticias internacionales: no encaja en las 4 categorías",
}

conteo_secciones = df["seccion_original"].value_counts().rename("noticias").rename_axis("seccion_original").reset_index()
equivalencias = pd.DataFrame(list(MAPEO.items()), columns=["seccion_original", "etiqueta"]).merge(
    conteo_secciones, on="seccion_original", how="left"
)
print("TABLA DE EQUIVALENCIAS")
display(equivalencias)

df["etiqueta"] = df["seccion_original"].map(MAPEO)
sin_categoria = df[df["etiqueta"].isna()]
tabla_descartes = (
    sin_categoria.assign(motivo=sin_categoria["seccion_original"].map(lambda s: MOTIVO_DESCARTE.get(s, "sección sin equivalencia")))
    .groupby(["seccion_original", "motivo"]).size().rename("noticias").reset_index()
)
print("SECCIONES DESCARTADAS")
display(tabla_descartes)

descartes["sección sin categoría equivalente"] = len(sin_categoria)
df = df.dropna(subset=["etiqueta"]).reset_index(drop=True)

TABLA DE EQUIVALENCIAS


,seccion_original,etiqueta,noticias
0,politica,politica,94
1,elecciones,politica,2
2,economia,economia,93
3,deporte-total,deportes,101
4,tecnologia,ciencia_tecnologia,95


SECCIONES DESCARTADAS


,seccion_original,motivo,noticias
0,bbc-news-mundo,noticias internacionales: no encaja en las 4 categorías,2
1,ecdata,periodismo de datos: tema mixto,4
2,eldominical,revista dominical: tema mixto,1
3,opinion,"columnas de opinión: tema mixto, sin categoría clara",4
4,ruedas-tuercas,motores: no encaja en las 4 categorías,2


Como control: la etiqueta sale de la sección de la URL, no del sitemap de origen. Esta tabla cruzada muestra cuántas notas cayeron en una categoría distinta a la del sitemap del que se descargaron.

In [4]:
pd.crosstab(df["feed"], df["etiqueta"])

etiqueta,ciencia_tecnologia,deportes,economia,politica
feed,,,,
deporte-total,1,101,0,0
economia,0,0,93,0
politica,0,0,0,96
tecnologia,94,0,0,0


## 3. Reporte de descartes

In [5]:
assert sum(descartes.values()) == len(raw) - len(df), "los descartes no suman"
reporte = pd.Series(descartes, name="noticias descartadas").to_frame()
reporte.loc["TOTAL"] = reporte.sum()
print(f"Crudas: {len(raw)} | Descartadas: {len(raw) - len(df)} | Finales: {len(df)}")
reporte

Crudas: 404 | Descartadas: 19 | Finales: 385


,noticias descartadas
titular o URL vacíos,0
URL duplicada,0
nota diaria 'Precio del dólar' (casi duplicada),6
titular repetido,0
sección sin categoría equivalente,13
TOTAL,19


## 4. Análisis exploratorio (EDA)

Los títulos de los gráficos se calculan con los datos, para que expresen la conclusión y no solo la variable.

In [6]:
conteo = df["etiqueta"].value_counts().reindex(CATEGORIAS).fillna(0).astype(int)
balanceado = (conteo.max() - conteo.min()) <= 0.25 * conteo.max()
titulo = (f"Las 4 categorías quedan {'balanceadas' if balanceado else 'desbalanceadas'}: "
          f"entre {conteo.min()} y {conteo.max()} noticias cada una")

fig1 = px.bar(conteo.rename_axis("etiqueta").reset_index(name="noticias"),
              x="etiqueta", y="noticias", color="etiqueta", text="noticias", title=titulo)
fig1.update_layout(showlegend=False, xaxis_title="Categoría", yaxis_title="Noticias")
fig1.show()

In [7]:
df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")
por_dia = (df.dropna(subset=["fecha"]).assign(dia=lambda d: d["fecha"].dt.strftime("%Y-%m-%d"))
             .groupby(["dia", "etiqueta"]).size().rename("noticias").reset_index())
total_dia = por_dia.groupby("dia")["noticias"].sum()
titulo = (f"El corpus cubre {total_dia.shape[0]} días; el {total_dia.idxmax()} concentra "
          f"más noticias ({total_dia.max()})")

fig2 = px.bar(por_dia, x="dia", y="noticias", color="etiqueta", title=titulo)
fig2.update_layout(xaxis_title="Fecha de publicación", yaxis_title="Noticias", legend_title="Categoría")
fig2.show()

In [8]:
df["largo"] = df["titular"].str.len()
q1, med, q3 = df["largo"].quantile([0.25, 0.5, 0.75])
titulo = f"La mitad de los titulares mide entre {q1:.0f} y {q3:.0f} caracteres (mediana: {med:.0f})"

fig3 = px.histogram(df, x="largo", nbins=30, title=titulo)
fig3.update_layout(xaxis_title="Caracteres del titular", yaxis_title="Noticias")
fig3.show()

## 5. Validación: titulares más largos y más cortos

In [9]:
cols = ["titular", "largo", "etiqueta"]
print("5 titulares MÁS LARGOS")
display(df.nlargest(5, "largo")[cols])
print("5 titulares MÁS CORTOS")
display(df.nsmallest(5, "largo")[cols])
print(f"Mínimo {df['largo'].min()} y máximo {df['largo'].max()} caracteres: son los extremos del histograma "
      f"(cola izquierda y cola derecha); la mediana es {med:.0f}.")

5 titulares MÁS LARGOS


,titular,largo,etiqueta
383,“El jugador peruano tiene un déficit grande en nutrición y táctica que resuelve a los ...,178,deportes
261,“Con Mano Menezes el mejor jugador de la selección ha sido un ecuatoriano”: Bruno Cava...,171,deportes
382,El furioso “Que se dedique a administrar” de Guede a Franco Velazco y un Quevedo que r...,171,deportes
232,“Creo que puedo llegar a un top 3 sin que nadie se quede botado”: Raúl ‘Mono’ Orlandin...,167,deportes
49,"Keiko Fujimori sobre Álvaro Vargas Llosa: “Confío en que su trayectoria, experiencia y...",166,politica


5 titulares MÁS CORTOS


,titular,largo,etiqueta
150,Invocación por el Perú,22,economia
95,AlaCena no tiene precio,23,economia
113,IA y Sandbox regulatorio,24,economia
163,La evolución de la confianza en los pagos,41,economia
97,Premio Nobel de Economía: la macro es micro,43,economia


Mínimo 22 y máximo 178 caracteres: son los extremos del histograma (cola izquierda y cola derecha); la mediana es 98.


**Comprobación de URLs.** Se abre una muestra aleatoria de 15 enlaces (con pausa de 1.5 s) para confirmar que funcionan (código 200).

In [10]:
estados = []
for url in df["url"].sample(min(15, len(df)), random_state=42):
    try:
        estados.append(requests.get(url, headers={"User-Agent": UA}, timeout=20).status_code)
    except requests.RequestException:
        estados.append("error")
    time.sleep(1.5)
pd.Series(estados, name="código HTTP").value_counts().to_frame()

,count
código HTTP,
200,15


## 6. Exportación

In [11]:
COLUMNAS = ["titular", "bajada", "medio", "seccion_original", "url", "fecha", "etiqueta"]
final = df.assign(fecha=df["fecha"].dt.strftime("%Y-%m-%d"))[COLUMNAS]
final.to_csv("datos/noticias.csv", index=False)

print("df.shape:", final.shape, "| bajadas vacías:", (final["bajada"] == "").sum())
if len(final) < 350:
    print("ATENCIÓN: hay menos de 350 noticias; el mínimo del enunciado es 350.")
final.head()

df.shape: (385, 7) | bajadas vacías: 0


,titular,bajada,medio,seccion_original,url,fecha,etiqueta
0,Facultades legislativas: Senado sigue sin alcanzar consenso y debate continúa entrampado,El pedido de facultades legislativas del Ejecutivo continúa entrampado en el Senado po...,El Comercio,politica,https://elcomercio.pe/politica/facultades-legislativas-senado-sigue-sin-alcanzar-conse...,2026-10-17,politica
1,Renovación Popular se declara en reorganización partidaria: ¿A qué se debe y cuáles so...,"La medida deja sin efecto las credenciales de los coordinadores regionales, provincial...",El Comercio,politica,https://elcomercio.pe/politica/renovacion-popular-se-declara-en-reorganizacion-partida...,2026-10-17,politica
2,"Alfonso, ¿ahora qué?, una crónica de Fernando Vivas sobre López-Chau, líder de Ahora N...","El líder de Ahora Nación y portavoz de su bancada de senadores, Alfonso López-Chau, qu...",El Comercio,politica,https://elcomercio.pe/politica/alfonso-ahora-que-una-cronica-de-fernando-vivas-sobre-l...,2026-10-17,politica
3,Tomás Gálvez dice que Ministerio Público requiere un crédito suplementario: “No tendrí...,"El fiscal de la Nación, Tomás Gálvez, pidió al Gobierno un crédito suplementario de ur...",El Comercio,politica,https://elcomercio.pe/politica/actualidad/tomas-galvez-dice-que-ministerio-publico-req...,2026-10-17,politica
4,De Gregorio Santos a “El lunarejo”: más de 8 mil condenados solicitaron el indulto pre...,"Entre los solicitantes se encuentra Gregorio Santos, exgobernador de Cajamarca, conden...",El Comercio,politica,https://elcomercio.pe/politica/indultos-presidenciales-lista-gracias-presidenciales-ke...,2026-10-17,politica
